# Routing ????????? checkpoint?

?? notebook ?????????????
- ?? best checkpoint
- ?? norm / denorm ??
- ??????????????

In [ ]:
from pathlib import Path
import sys
import json
import yaml

import torch
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
from datasets import build_dataset, build_dataloader
from models import build_model
from trainers import Trainer, build_loss, set_seed

In [ ]:
TRAIN_CFG_PATH = PROJECT_ROOT / "configs" / "train.yaml"
DATA_CFG_PATH = PROJECT_ROOT / "configs" / "data.yaml"
MODEL_CFG_PATH = PROJECT_ROOT / "configs" / "model.yaml"

def load_yaml(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

def merge_model_cfg(model_cfg_file: dict, train_cfg: dict):
    cfg = {}
    if isinstance(model_cfg_file.get("model"), dict):
        cfg.update(model_cfg_file["model"])
    if isinstance(train_cfg.get("model"), dict):
        cfg.update(train_cfg["model"])
    return cfg

train_cfg = load_yaml(TRAIN_CFG_PATH)
model_cfg_file = load_yaml(MODEL_CFG_PATH)
model_cfg = merge_model_cfg(model_cfg_file, train_cfg)

trainer_cfg = dict(train_cfg.get("trainer", {}))
checkpoint_dir = PROJECT_ROOT / trainer_cfg.get("checkpoint_dir", "checkpoints/routing_baseline")
CHECKPOINT_PATH = checkpoint_dir / "best.ckpt"
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

In [ ]:
set_seed(int(train_cfg.get("seed", 42)))
dataset_cfg = dict(train_cfg.get("dataset", {}))
dataset_cfg.setdefault("data_cfg_path", str(DATA_CFG_PATH))
dataset_cfg.setdefault("model_cfg_path", str(MODEL_CFG_PATH))

train_ds = build_dataset("train", dataset_kwargs=dataset_cfg)
eval_kwargs = dict(dataset_cfg)
eval_kwargs["normalizers"] = train_ds.normalizers
test_ds = build_dataset("test", dataset_kwargs=eval_kwargs)

eval_loader_cfg = dict(train_cfg.get("dataloader", {}).get("val", {}))
eval_loader_cfg["shuffle"] = False
eval_loader_cfg["use_balance_sampler"] = False
test_loader = build_dataloader(test_ds, **eval_loader_cfg)

if "pred_len" not in model_cfg and "n_pred" in dataset_cfg:
    model_cfg["pred_len"] = dataset_cfg["n_pred"]
model = build_model(model_cfg=model_cfg, dataset=train_ds)
criterion = build_loss(train_cfg.get("loss", {}))

trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=None,
    scheduler=None,
    device=train_cfg.get("device", "auto"),
    checkpoint_dir=str(checkpoint_dir),
    monitor=str(trainer_cfg.get("monitor", "val_loss")),
    monitor_mode=str(trainer_cfg.get("monitor_mode", "min")),
)
trainer.load_checkpoint(str(CHECKPOINT_PATH), strict=True)

In [ ]:
def compute_metrics(pred: torch.Tensor, target: torch.Tensor):
    err = pred - target
    mse = torch.mean(err * err)
    mae = torch.mean(torch.abs(err))
    rmse = torch.sqrt(mse + 1e-12)
    denom = torch.sum((target - torch.mean(target)) ** 2)
    nse = 1.0 - torch.sum(err ** 2) / (denom + 1e-12)
    return {
        "loss": float(mse.detach().cpu()),
        "mae": float(mae.detach().cpu()),
        "rmse": float(rmse.detach().cpu()),
        "nse": float(nse.detach().cpu()),
    }

pred_norm, target_norm = trainer.predict(test_loader, return_target=True)
pred_denorm = test_ds.inverse_transform_streamflow_tensor(pred_norm)
target_denorm = test_ds.inverse_transform_streamflow_tensor(target_norm)

metrics_norm = compute_metrics(pred_norm, target_norm)
metrics_denorm = compute_metrics(pred_denorm, target_denorm)

print("metrics_norm:")
print(json.dumps(metrics_norm, ensure_ascii=False, indent=2))
print("metrics_denorm:")
print(json.dumps(metrics_denorm, ensure_ascii=False, indent=2))

In [ ]:
# ?????????
num_outlets = pred_denorm.shape[-1]
names = getattr(test_ds, "outlet_names", [f"outlet_{i}" for i in range(num_outlets)])

for i in range(num_outlets):
    p = pred_denorm[:, 0, i].detach().cpu().numpy()
    t = target_denorm[:, 0, i].detach().cpu().numpy()

    plt.figure(figsize=(4, 4))
    plt.scatter(t, p, s=8, alpha=0.35)
    lo = float(np.nanmin([t.min(), p.min()]))
    hi = float(np.nanmax([t.max(), p.max()]))
    plt.plot([lo, hi], [lo, hi], 'r--', linewidth=1)
    plt.xlabel("obs (denorm)")
    plt.ylabel("pred (denorm)")
    plt.title(f"Scatter: {names[i]}")
    plt.tight_layout()
    plt.show()

In [ ]:
# ??? + ???????????
max_points = 500
for i in range(num_outlets):
    p = pred_denorm[:, 0, i].detach().cpu().numpy()[:max_points]
    t = target_denorm[:, 0, i].detach().cpu().numpy()[:max_points]
    e = p - t

    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    axes[0].plot(t, label="obs", linewidth=1.5)
    axes[0].plot(p, label="pred", linewidth=1.2)
    axes[0].set_title(f"Time Series: {names[i]}")
    axes[0].legend()

    axes[1].hist(e, bins=40, alpha=0.8)
    axes[1].set_title(f"Error Histogram: {names[i]}")
    axes[1].set_xlabel("pred - obs")

    plt.tight_layout()
    plt.show()